# Analysis: the tables and figures of the paper

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Avi-Seth/Energy-Accuracy-Tradeoff-Benchmarking-on-Spatio-Temporal-Datasets/blob/main/04_analyze_results.ipynb)

This notebook reads the runs saved by `02_train_SHD.ipynb` and `03_train_SSC.ipynb` and makes every
table and figure. It needs no GPU (Runtime > Change runtime type > CPU is enough).

**How to run it.** Runtime > Run all. As it is set, the notebook downloads the runs of the study
(`results.zip`, 1.5 MB, from the GitHub repository of these notebooks) and remakes every table and
figure in about a minute. To analyse your own runs, set `RESULTS = 'mine'` in Step 1; the notebook then
reads them from your Google Drive, and can be run as soon as one training notebook has finished
(whatever has not been run yet is skipped).

**How to read the code.** Every code cell starts with a short explanation above it, and the text after
a `#` on a line of code says what that line does.

**What it computes.** For every setting, the **order-blind ceiling r** is the order-blind design (the
count classifier excluded) with the highest average accuracy on the held-out recordings over the seeds;
the test recordings never choose anything. The LSTM's input-dropout level is chosen on the held-out
recordings the same way. For every order-using network, seed by seed:

```
a0 = test accuracy with the windows in order
a1 = test accuracy of a copy trained and tested on shuffled windows
drop = a0 - a1,   gain = a0 - r,   deficit = r - a1,   so drop = gain + deficit
```

**The decision rule.** A quantity measured once per seed "passes" when its average over the seeds is
more than twice its standard deviation (paired by seed). With five seeds this is the same as t > 4.47,
with three seeds t > 3.46.

- **H2 verdict** for every network: the gain passes: "may need order at this width, by at most the
  gain"; the average gain is positive but does not pass: "not supported"; the average gain is zero or
  negative: "reachable without order at this width"; training accuracy below halfway from chance to
  perfect: "not interpreted".
- **H1**: for every competent network whose drop passes, the deficit passes too (the loss from
  scrambling is then larger than what order can add for that network).

Energy per recording (computed, not measured) comes from the runs. Models are compared within one of
the three cases (worst, best within a window, best across the windows); a conclusion is robust only if
it holds in all three.

**Which designs each network is measured against.** On SHD at 25 windows, the 128-unit networks are
measured against the 7 order-blind designs of 128 to 512 units, and RadLIF against all 8 (adding the
design at RadLIF's scale); Table S1 sets every 25-window network beside both. (The count classifier is
never a candidate: it is the floor.) At 14 ms and on
SSC, every network is measured against every design.

**Outputs.** Every table is shown here and saved as `.csv` and `.md`; every figure is shown and saved as
`.png` (300 dpi) and `.pdf`; `summary.json` holds every verdict.

## Step 1. Settings

- `RESULTS = 'published'` analyses the runs of the study: `results.zip`, downloaded from `RESULTS_URL`
  (the GitHub repository of these notebooks). `RESULTS = 'mine'` analyses your own runs in
  `MY_RESULTS_DIR` on your Google Drive (for a quick test, the folder `results_quick/` instead of
  `results/`).

In [ ]:
RESULTS = 'published'                         # 'published': the study's runs; 'mine': your own runs on Drive
RESULTS_URL = ('https://raw.githubusercontent.com/Avi-Seth/'
               'Energy-Accuracy-Tradeoff-Benchmarking-on-Spatio-Temporal-Datasets/main/results.zip')   # results.zip in
                                              # the GitHub repository of these notebooks
MY_RESULTS_DIR = '/content/drive/MyDrive/shd_ssc_order/results/'   # your own runs (with RESULTS = 'mine')

## Step 2. Get the runs

With `RESULTS = 'published'`, `results.zip` is downloaded from `RESULTS_URL` and unpacked on the Colab
machine. If you uploaded `results.zip` yourself to the Colab machine (the folder icon on the left, then
the upload icon), that copy is used instead. With `RESULTS = 'mine'`, Google Drive is connected (Colab
asks once for permission) and the runs are read there. The tables and figures go to `OUT_DIR`.

In [ ]:
import os                                     # folders and file names
import shutil                                 # copying files and making .zip files
import zipfile                                # unpacking results.zip
import urllib.request                         # downloading results.zip

if RESULTS == 'mine':
    from google.colab import drive            # the tool that connects Google Drive to this notebook
    drive.mount('/content/drive')             # your Drive files appear under /content/drive/MyDrive
    RESULTS_DIR = MY_RESULTS_DIR                                  # the runs of the training notebooks
    OUT_DIR = MY_RESULTS_DIR + 'analysis/'                        # tables and figures go here, on your Drive
else:
    PUBLISHED_DIR = '/content/shd_ssc_order/published/'           # on this Colab machine
    RESULTS_DIR = PUBLISHED_DIR + 'results/'                      # results.zip unpacks into this folder
    OUT_DIR = '/content/shd_ssc_order/analysis/'                  # tables and figures go here
    if not os.path.exists(RESULTS_DIR):                           # not unpacked yet in this session
        os.makedirs(PUBLISHED_DIR, exist_ok=True)
        archive = PUBLISHED_DIR + 'results.zip'
        if os.path.exists('/content/results.zip'):                # uploaded by hand: use that copy
            shutil.copy('/content/results.zip', archive)
        else:
            try:
                print('downloading', RESULTS_URL)
                urllib.request.urlretrieve(RESULTS_URL, archive)
            except Exception as error:                            # no connection, a moved file, ...
                raise RuntimeError(f'results.zip could not be downloaded from {RESULTS_URL} ({error}). Check '
                                   f'RESULTS_URL in Step 1, or upload results.zip to this Colab machine (the '
                                   f'folder icon on the left, then the upload icon) and run the notebook '
                                   f'again.') from None
        with zipfile.ZipFile(archive) as z:
            z.extractall(PUBLISHED_DIR + 'unpacking/')            # unpack under a temporary folder first
        os.replace(PUBLISHED_DIR + 'unpacking/results', RESULTS_DIR)   # the real name, now that it is complete
        os.rmdir(PUBLISHED_DIR + 'unpacking/')                   # the temporary folder is now empty
print('runs read from', RESULTS_DIR)
print('tables and figures go to', OUT_DIR)

## Step 3. Libraries and figure style

pandas shows the tables; matplotlib draws the figures. The three colours were checked for colour-blind
readers, and every colour also comes with its own marker shape or a label.

In [ ]:
import glob                                   # finding all run files in a folder
import json                                   # reading the run files
import math                                   # square roots and rounding
import numpy as np                            # arrays of numbers
import pandas as pd                           # tables
import matplotlib.pyplot as plt               # figures
from matplotlib.lines import Line2D           # legend entries
from matplotlib.patches import FancyBboxPatch, Patch    # boxes (Figure 1) and coloured legend entries
from matplotlib.ticker import FuncFormatter, LogLocator, NullFormatter   # labels of the log-scale axes
from matplotlib import patheffects                # a white outline that keeps text readable over lines (Figure 4)
from matplotlib.transforms import Bbox, blended_transform_factory   # label boxes (Figure 3); text at the right edge (Figure 2)
from IPython.display import display           # shows a table in the notebook

os.makedirs(OUT_DIR, exist_ok=True)           # create the output folder if it does not exist yet
pd.set_option('display.max_colwidth', None)   # show long table entries in full, not cut off with '...'

INK, INK2, MUTED, GRID = '#0b0b0b', '#52514e', '#898781', '#e1e0d9'    # text, secondary text, grey, grid
BLUE, ORANGE, AQUA = '#2a78d6', '#eb6834', '#1baf7a'                    # the three colours
FAMILY_STYLE = {'counts only': (MUTED, 's'),          # grey square
                'order-blind': (BLUE, 'o'),           # blue circle
                'conventional': (ORANGE, '^'),        # orange triangle
                'spiking': (AQUA, 'D')}               # green diamond
plt.rcParams.update({
    'font.family': 'DejaVu Sans',                                          # matplotlib's own font: the same on every computer
    'font.size': 7.5, 'axes.titlesize': 8, 'axes.labelsize': 7.5,         # small fonts: the figures are
    'xtick.labelsize': 7, 'ytick.labelsize': 7, 'legend.fontsize': 7,      # printed at their real size
    'axes.edgecolor': '#c3c2b7', 'axes.labelcolor': INK2,                  # a light grey frame, dark grey labels
    'xtick.color': INK2, 'ytick.color': INK2,
    'axes.grid': True, 'grid.color': GRID, 'grid.linewidth': 0.5,          # a thin light grid ...
    'axes.axisbelow': True,                                                # ... drawn behind the data
    'axes.spines.top': False, 'axes.spines.right': False,                  # no frame line on the top and right
    'figure.facecolor': 'white', 'savefig.facecolor': 'white',             # white background
    'pdf.fonttype': 42})                                                   # keeps the text editable in the PDF

## Step 4. The settings of the analysis

- `GROUPS`: the four settings, and which networks are measured against which designs.
- `SAME_SIZE`: every network's order-blind partner with the same number of weights.
- `PANELS` and `LADDER`: what Figures 3, 4 and S1 draw.
- the bounds of the control checks: the best order-using network must beat 0.40 on the chains and 0.90
  on the reversal task, and every drop on the negative control must stay below 0.04.

In [ ]:
LSTM_LEVELS = ['lstm_d0', 'lstm_d20', 'lstm_d50', 'lstm_d70']    # the LSTM's four input-dropout levels

# Every setting: a short name ('key'), its title in the tables ('label'), the experiment folder its runs
# are in, which order-blind designs it is measured against ('128 units' or 'all'), its order-using
# networks, and which network is its LSTM. 'lstm' stands for the dropout level chosen on held-out data.
GROUPS = [
    {'key': 'shd_25', 'label': 'SHD, 25 windows', 'experiment': 'shd_25', 'candidates': '128 units',
     'networks': ['lif_std', 'rlif', 'rnn', 'lstm'], 'lstm': 'lstm'},
    {'key': 'shd_25_radlif', 'label': 'SHD, 25 windows, RadLIF scale', 'experiment': 'shd_25', 'candidates': 'all',
     'networks': ['radlif'], 'lstm': 'lstm'},
    {'key': 'shd_14ms', 'label': 'SHD, 14 ms windows', 'experiment': 'shd_14ms', 'candidates': 'all',
     'networks': ['radlif', 'rlif', 'lstm'], 'lstm': 'lstm'},
    {'key': 'ssc_10ms', 'label': 'SSC, 10 ms windows', 'experiment': 'ssc_10ms', 'candidates': 'all',
     'networks': ['lstm_paper', 'lif_std', 'rlif', 'rnn'], 'lstm': 'lstm_paper'},
]
SAME_SIZE = {'lif_std': 'perminv',            # network -> the order-blind design with its number of weights
             'rlif': 'pool_128', 'rnn': 'pool_128',
             'lstm_paper': 'pool_512',
             'radlif': 'ob_deep_1024_d10',
             'lstm_d0': 'pool_512', 'lstm_d20': 'pool_512', 'lstm_d50': 'pool_512', 'lstm_d70': 'pool_512'}
PANELS = [                                    # Figure 3: one panel per dataset and width
    {'title': 'SHD, 25 windows per recording', 'groups': ['shd_25', 'shd_25_radlif']},
    {'title': 'SHD, 14 ms windows', 'groups': ['shd_14ms']},
    {'title': 'SSC, 10 ms windows', 'groups': ['ssc_10ms']},
]
LADDER = [                                    # Figure S1: the order-blind designs, from simple to rich
    ['count_256'],
    ['perminv', 'pool_128', 'pool_512'],
    ['meanmax_deep_128', 'meanmax_deep_128_d20', 'meanmax_deep_128_d50', 'meanmax_deep_128_d70'],
    ['ob_deep_1024_d10'],
]
POSITIVE_CONTROL_BOUND = {'chain': 0.40, 'reversal': 0.90}   # the best order-using network must beat these
NEGATIVE_CONTROL_BOUND = 0.04                                 # no drop on the negative control may reach this

## Step 5. Read the runs

Every experiment folder holds `meta.json` (what the experiment is) and a folder `runs` with one file per
run. An experiment is kept here as a dictionary with its description, its seeds and its runs. A run is
found by (task, model, q, seed): task is only used by the control tasks (None elsewhere), and q = 1
marks the copy trained and tested on shuffled windows.

The small functions below answer the questions the tables ask: are all seeds of a model there? What
are its accuracies seed by seed? What is its average energy?

In [ ]:
def read_experiment(folder):
    """Read one experiment folder: its description (meta.json) and every run file in runs/."""
    with open(folder + 'meta.json') as f:                     # open the description ...
        meta = json.load(f)                                   # ... and read it into a dictionary
    runs = {}                                                 # every run, found by (task, model, q, seed)
    for path in sorted(glob.glob(folder + 'runs/*.json')):    # every run file of the folder
        with open(path) as f:
            r = json.load(f)                                  # one run: accuracies, answers, energy, ...
        key = (r.get('task'), r['model'], r['q'], r['seed'])  # task is None outside the control tasks
        runs[key] = r
    return {'name': meta['experiment'], 'meta': meta, 'seeds': meta['seeds'], 'runs': runs}


def has(exp, model, q=0, task=None):
    """True when the model has a run for every seed (q = 0: windows in order; q = 1: shuffled)."""
    for seed in exp['seeds']:
        if (task, model, q, seed) not in exp['runs']:         # one missing seed is enough to say no
            return False
    return True


def per_seed(exp, model, field='test_acc', q=0, task=None):
    """One number per seed, in the order of the seeds (test accuracy unless another field is asked for)."""
    values = []
    for seed in exp['seeds']:
        values.append(exp['runs'][(task, model, q, seed)][field])
    return np.array(values, dtype=np.float64)                 # an array, so two models subtract seed by seed


def mean_energy(exp, model):
    """Average energy per recording over the seeds (windows in order), in pJ, in the three cases:
    [worst, best within a window, best across the windows]."""
    rows = []
    for seed in exp['seeds']:
        e = exp['runs'][(None, model, 0, seed)]['energy_pJ']  # the energy saved with this run
        rows.append([e['worst'], e['within'], e['across']])
    return np.array(rows).mean(0)                             # the average over the seeds (the rows)


def designs_of(exp):
    """The order-blind designs of an experiment that have a run for every seed."""
    found = []
    for name in exp['meta']['designs']:
        if has(exp, name):
            found.append(name)
    return found


exps = {}                                                     # experiment name -> its description and runs
for name in ['shd_controls', 'shd_25', 'shd_14ms', 'ssc_10ms']:
    folder = RESULTS_DIR + name + '/'
    if os.path.exists(folder + 'meta.json'):                  # skip the experiments not run yet
        exps[name] = read_experiment(folder)
        print(f'{name}: {len(exps[name]["runs"])} runs')
    else:
        print(f'{name}: no results yet')

## Step 6. The decision rule, the verdicts and the decomposition

- `rule` gives the average, the standard deviation, t, and whether the quantity passes.
- `verdict` turns a network's gain into its H2 verdict.
- `choose` picks the design with the highest average held-out accuracy (the first one wins an exact tie).
- `decompose` splits one network's drop into gain and deficit, seed by seed.
- `analyse` does all of this for every setting of `GROUPS`.

In [ ]:
def rule(d):
    """The decision rule for one quantity measured once per seed (for example a0 - r).
    Returns (average, standard deviation, t, passes)."""
    d = np.asarray(d, dtype=np.float64)
    m = float(d.mean())                           # the average over the seeds
    if len(d) > 1:
        sd = float(d.std(ddof=1))                 # the sample standard deviation (it divides by n - 1)
    else:
        sd = 0.0                                  # one seed: no spread can be measured
    if sd > 1e-12:
        t = m / (sd / math.sqrt(len(d)))          # t = average / (sd / square root of the number of seeds)
    else:
        t = float('nan')                          # no spread at all: t is undefined
    return m, sd, t, bool(m > 2 * sd)             # passes: the average is more than twice the sd


def verdict(gain, competent):
    """The H2 verdict of one network, from its gain (a rule result) and whether it trained well enough."""
    if not competent:
        return 'not interpreted'                  # trained below the competence bar
    if gain[3]:
        return 'may need order at this width, by at most the gain'   # the gain passes the rule
    if gain[0] > 0:
        return 'not supported'                    # positive on average, but not clearly
    return 'reachable without order at this width'   # zero or negative on average


def choose(exp, candidates, field='val_acc'):
    """The candidate with the highest average over the seeds (held-out accuracy unless another field is
    asked for). When two are exactly equal, the first one in the list wins."""
    best, best_value = None, None
    for name in candidates:
        value = per_seed(exp, name, field).mean()
        if best is None or value > best_value:    # strictly higher: an exact tie keeps the earlier one
            best, best_value = name, value
    return best


def candidates_of(exp, kind):
    """The designs a network is measured against: kind is '128 units' (the designs of 128 to 512 units)
    or 'all'. The count classifier is never a candidate: it is the 'counts only' floor."""
    names = []
    for name in designs_of(exp):
        if name == 'count_256':
            continue                              # skip the count classifier
        if kind == '128 units' and exp['meta']['models'][name]['scale'] != '128 units':
            continue                              # skip the designs at RadLIF's scale
        names.append(name)
    return names


def resolve(exp, name):
    """The model a network name stands for, or None when its runs are missing. 'lstm' stands for the
    input-dropout level chosen on held-out recordings."""
    if name == 'lstm':
        levels = []                               # the LSTM levels with every run, in order and shuffled
        for level in LSTM_LEVELS:
            if has(exp, level) and has(exp, level, q=1):
                levels.append(level)
        if len(levels) == 0:
            return None
        return choose(exp, levels)                # chosen on held-out accuracy, never on test accuracy
    if has(exp, name) and has(exp, name, q=1):
        return name
    return None


def decompose(exp, model, r_seeds, bar):
    """One network: a0, a1, training accuracy, competence, gain, drop, deficit, H2 verdict and H1."""
    a0 = per_seed(exp, model)                     # test accuracy, windows in order (one per seed)
    a1 = per_seed(exp, model, q=1)                # test accuracy, retrained on shuffled windows
    train = float(per_seed(exp, model, 'train_acc').mean())   # average training accuracy
    competent = train >= bar                      # at least halfway from chance to perfect
    gain = rule(a0 - r_seeds)                     # gain = a0 - r, seed by seed
    drop = rule(a0 - a1)                          # drop = a0 - a1
    deficit = rule(r_seeds - a1)                  # deficit = r - a1
    eligible = competent and drop[3]              # H1 is tested only when the drop passes
    if eligible:
        h1_holds = deficit[3]                     # H1 holds when the deficit passes too
    else:
        h1_holds = None                           # not tested
    return {'model': model, 'a0': a0, 'a1': a1, 'train': train,
            'train_shuffled': float(per_seed(exp, model, 'train_acc', q=1).mean()), 'competent': competent,
            'gain': gain, 'drop': drop, 'deficit': deficit, 'verdict': verdict(gain, competent),
            'h1_eligible': eligible, 'h1_holds': h1_holds}


def analyse(exps):
    """Every setting of GROUPS: its candidates, its ceiling, the best design on test (a check only), its
    LSTM, and every network decomposed."""
    A = {}
    for g in GROUPS:
        if g['experiment'] not in exps:
            continue                              # this experiment has not been run yet
        exp = exps[g['experiment']]
        cands = candidates_of(exp, g['candidates'])
        if len(cands) == 0:
            continue                              # no order-blind design has every seed yet
        ceiling = choose(exp, cands)              # the ceiling: chosen on held-out recordings
        best_test = choose(exp, cands, 'test_acc')   # a check only: never used to choose anything
        r = per_seed(exp, ceiling)                # the ceiling's test accuracy, one per seed
        nets = {}
        for n in g['networks']:
            m = resolve(exp, n)
            if m is not None:                     # skip networks whose runs are missing
                nets[n] = decompose(exp, m, r, exp['meta']['competence_bar'])
        A[g['key']] = {'group': g, 'exp': exp, 'candidates': cands, 'ceiling': ceiling, 'r': r,
                       'best_test': best_test, 'networks': nets, 'lstm': resolve(exp, g['lstm'])}
    return A


A = analyse(exps)                                 # A holds everything the tables and figures need
for key in A:
    print(f'{A[key]["group"]["label"]}: ceiling {A[key]["ceiling"]} (chosen among {len(A[key]["candidates"])} designs)')

## Step 7. Writing tables and figures

Numbers are written as percentages with one decimal (`pct`), as differences in points with their t
(`pts`), and energies in microjoules (`uj`). `show_table` shows a table and saves it as .csv and .md;
`save_figure` saves a figure as .png and .pdf and shows it.

In [ ]:
def minus(s):
    """Replace '-' by the typographic minus sign used in printed tables."""
    return s.replace('-', '−')


def pct(x, sd=None):
    """A share as a percentage with one decimal: 0.8123 -> '81.2'; with its sd: '81.2 ± 1.3'."""
    if sd is None:
        return f'{100 * x:.1f}'
    return f'{100 * x:.1f} ± {100 * sd:.1f}'      # ± is the plus-minus sign


def pts(q):
    """A rule result (average, sd, t, passes) in points, with its sign and its t: '+2.4 ± 1.2 (4.3)'."""
    m, sd, t, _ = q
    if math.isnan(t):
        tt = 'undefined'                          # no spread: t cannot be computed
    else:
        tt = f'{t:.1f}'
    return minus(f'{100 * m:+.1f} ± {100 * sd:.1f} ({tt})')


def uj(p):
    """pJ -> microjoules, with three significant digits."""
    return f'{p / 1e6:.3g}'


def met(ok):
    """How a check is reported in Table 5."""
    if ok:
        return 'met'
    return 'NOT MET'


def show_table(rows, name, title):
    """Show a table (a list of rows, each a dictionary of column -> value) and save it as .csv and .md."""
    if len(rows) == 0:
        print(name + ': nothing to show (the runs it needs are missing)')
        return
    df = pd.DataFrame(rows)                       # the rows become a pandas table
    df.to_csv(OUT_DIR + name + '.csv', index=False)   # for spreadsheets
    lines = ['**' + title + '**', '', '| ' + ' | '.join(df.columns) + ' |', '|' + '---|' * len(df.columns)]
    for _, row in df.iterrows():                  # one line of the Markdown table per row
        lines.append('| ' + ' | '.join(str(v).replace('|', '/') for v in row) + ' |')
    with open(OUT_DIR + name + '.md', 'w') as f:  # a Markdown table, for the paper or for GitHub
        f.write('\n'.join(lines) + '\n')
    print(title)
    display(df)                                   # show the table in the notebook


def save_figure(fig, name):
    """Save a figure as .png (300 dots per inch) and as .pdf, then show it."""
    fig.savefig(OUT_DIR + name + '.png', dpi=300, bbox_inches='tight')
    fig.savefig(OUT_DIR + name + '.pdf', bbox_inches='tight')
    plt.show()

## Table 1. The order-blind side of every setting

Counts only (the count classifier), the ceiling (chosen on held-out recordings) and, as a check, the
design the test set would have chosen.

In [ ]:
rows = []
for key in A:                                     # one row per setting
    a = A[key]
    exp = a['exp']
    row = {'setting': a['group']['label']}
    if has(exp, 'count_256'):                     # the count classifier: spike counts only, no timing
        c = per_seed(exp, 'count_256')
        row['counts only, %'] = pct(c.mean(), c.std(ddof=1))
    else:
        row['counts only, %'] = ''
    row['ceiling design (candidates)'] = f'{a["ceiling"]} ({len(a["candidates"])})'
    row['ceiling r, %'] = pct(a['r'].mean(), a['r'].std(ddof=1))
    if a['best_test'] == a['ceiling']:            # would the test set have chosen the same design?
        row['best design on test (a check), %'] = 'the same design'
    else:
        row['best design on test (a check), %'] = f'{a["best_test"]}: {pct(per_seed(exp, a["best_test"]).mean())}'
    rows.append(row)
show_table(rows, 'table1_order_blind_side', 'Table 1. The order-blind side of every setting (mean ± sd over seeds).')

## Table 2. Every network against its ceiling

a0, the gain over the ceiling with its t, the H2 verdict, and H1 (drop = gain + deficit). A network
marked * trained below the competence bar.

In [ ]:
def h1_text(d):
    """What Table 2 says about H1 for one network."""
    if not d['competent']:
        return 'not interpreted'
    if not d['h1_eligible']:                      # the drop does not pass: H1 says nothing about it
        return minus(f'outside (drop {100 * d["drop"][0]:.1f}, t {d["drop"][2]:.1f})')
    text = minus(f'{100 * d["drop"][0]:.1f} = {100 * d["gain"][0]:+.1f} + {100 * d["deficit"][0]:.1f}')
    if not d['h1_holds']:
        text += ' (H1 FAILS)'
    return text


def star(d):
    """The network's name, with * when it trained below the competence bar."""
    if d['competent']:
        return d['model']
    return d['model'] + '*'


# readable names for the figures: the kind of model, its layers and its neurons (Figures 2 and 4)
NICE = {'count_256': 'counts only', 'perminv': 'OB scores 128', 'pool_128': 'OB mean 128',
        'pool_512': 'OB mean 512', 'ob_deep_1024_d10': 'OB 2×1,024', 'lif_std': 'SNN-FF 128',
        'rlif': 'SNN-R 128', 'rnn': 'RNN 128', 'lstm': 'LSTM 128', 'lstm_paper': 'LSTM 128',
        'radlif': 'RadLIF 2×1,024'}


def nice(model):
    """A readable name: the kind of model, its layers and its neurons."""
    if model.startswith('lstm_d'):
        return 'LSTM 128'                         # the LSTM at one level of dropout
    if model.startswith('meanmax_deep_128'):
        return 'OB 2×128'                         # the mean-and-maximum design at one level of dropout
    return NICE.get(model, model)


rows = []
for key in A:
    a = A[key]
    for n in a['networks']:                       # one row per network of the setting
        d = a['networks'][n]
        row = {}
        row['setting'] = a['group']['label']
        row['network'] = star(d)
        row['a0, %'] = pct(d['a0'].mean(), d['a0'].std(ddof=1))
        row['gain over r, points (t)'] = pts(d['gain'])
        row['gain passes'] = d['gain'][3] and d['competent']
        row['verdict'] = d['verdict']
        row['H1: drop = gain + deficit'] = h1_text(d)
        rows.append(row)
show_table(rows, 'table2_verdicts', 'Table 2. Every network against its ceiling. *Below the competence bar.')

## Table 3. Every network beside the order-blind design of its size

The lead of a network over the design with its number of weights, and their energy ratio in the three
cases ("robust" when the network is cheaper in all three cases, or dearer in all three).

In [ ]:
rows, seen = [], []
for key in A:
    a = A[key]
    exp = a['exp']
    for n in a['networks']:
        d = a['networks'][n]
        design = SAME_SIZE.get(d['model'])        # the order-blind design with this network's size
        if design is None or not has(exp, design):
            continue                              # no partner, or its runs are missing
        if (exp['name'], d['model']) in seen:
            continue                              # a network in two settings of one experiment: show it once
        seen.append((exp['name'], d['model']))
        lead = rule(d['a0'] - per_seed(exp, design))                      # the lead, seed by seed
        ratio = mean_energy(exp, d['model']) / mean_energy(exp, design)   # energy ratio in the 3 cases
        dearer = [x > 1 for x in ratio]           # in each case: is the network dearer than its partner?
        if all(dearer) or not any(dearer):        # all three cases agree
            robust = 'robust'
        else:
            robust = 'depends on the case'
        weights_net = exp['meta']['models'][d['model']]['weights']
        weights_design = exp['meta']['models'][design]['weights']
        row = {}
        row['setting'] = a['group']['label']
        row['network'] = star(d)
        row['weights (network; design)'] = f'{weights_net:,}; {weights_design:,}'   # 92308 -> 92,308
        row['same-size design: test %'] = f'{design}: {pct(per_seed(exp, design).mean())}'
        row['lead, points (t)'] = pts(lead)
        row['lead passes'] = lead[3] and d['competent']
        row['energy, network over design (worst / within / across)'] = ' / '.join(f'{x:.2f}' for x in ratio)
        row['energy comparison'] = robust
        rows.append(row)
show_table(rows, 'table3_same_size', 'Table 3. Each network beside the order-blind design with its number of weights. *Below the competence bar.')

## Table 4. Energy per recording

The ceiling design and every network of every setting, in the three cases, and the ratio to the LSTM of
the setting (computed, not measured).

In [ ]:
rows = []
for key in A:
    a = A[key]
    exp, lstm = a['exp'], a['lstm']
    if lstm is not None:
        ref = mean_energy(exp, lstm)              # the LSTM's energy: the reference of the ratios
    else:
        ref = None                                # no LSTM in this setting yet
    items = [('ceiling design (' + a['ceiling'] + ')', a['ceiling'], a['r'])]   # (label, model, accuracies)
    for n in a['networks']:
        d = a['networks'][n]
        items.append((star(d), d['model'], d['a0']))
    for label, model, acc in items:
        e = mean_energy(exp, model)
        if ref is None:
            over = ''
        elif model == lstm:
            over = '1'                            # the LSTM against itself
        else:
            over = ' / '.join(f'{x:.2f}' for x in e / ref)
        row = {}
        row['setting'] = a['group']['label']
        row['model'] = label
        row['accuracy, %'] = pct(acc.mean())
        row['energy, uJ (worst / within / across)'] = ' / '.join(uj(x) for x in e)
        row['over the LSTM of the setting'] = over
        rows.append(row)
show_table(rows, 'table4_energy', 'Table 4. Energy per test recording (computed, not measured) and its ratio to the LSTM of the setting.')

## Table 5. The control tasks

The answers are known in advance:

- on the chains and the reversal task, an order-blind model must score exactly chance in every seed
  and give one answer for all members of a group (the six chains of one triple, or a recording and its
  reversal), because they hold exactly the same windows;
- a network retrained on shuffled windows must fall to chance there (within 2 sd);
- the best order-using network must beat the bounds of step 4 (it can find order when it is there);
- on the negative control, no network may lose accuracy when its windows are shuffled once more.

In [ ]:
controls = None                                   # stays None when the control tasks have not been run
if 'shd_controls' in exps:
    exp = exps['shd_controls']
    rows, checks = [], []                         # the table, and every check with its result
    for task in exp['meta']['tasks']:             # chain, reversal, negative
        info = exp['meta']['tasks'][task]
        chance = info['chance']                   # 1 / number of classes
        groups = np.array(info['test_groups'])    # which test samples belong together

        # The order-blind designs
        for m in exp['meta']['designs']:
            if not has(exp, m, task=task):
                continue
            acc = per_seed(exp, m, task=task)
            row = {'task': task, 'model': m, 'side': 'order-blind', 'in order, %': pct(acc.mean(), acc.std(ddof=1)),
                   'shuffled, %': '', 'drop, points (t)': ''}
            if task == 'negative':                # no known answer for the designs here: only described
                row['check'] = 'none (no known answer)'
                row['result'] = 'described'
                rows.append(row)
                continue
            exact = bool(np.all(np.abs(acc - chance) < 5e-5))    # exactly chance in every seed
            one = True                                            # one answer for every member of a group?
            for seed in exp['seeds']:
                preds = np.array(exp['runs'][(task, m, 0, seed)]['preds'])   # the model's answers
                for g in np.unique(groups):
                    if len(set(preds[groups == g].tolist())) != 1:   # more than one answer in this group
                        one = False
            ok = exact and one
            checks.append((f'{task}: {m} exactly chance with one answer per group', ok))
            row['check'] = f'exactly {chance:.4f} in every seed, one answer per group'
            row['result'] = met(ok)
            rows.append(row)

        # The order-using networks
        best = None                               # (name, accuracy) of the best network so far
        for m in exp['meta']['networks']:
            if not (has(exp, m, task=task) and has(exp, m, 1, task=task)):
                continue
            a0 = per_seed(exp, m, task=task)              # windows in order
            a1 = per_seed(exp, m, q=1, task=task)         # retrained on shuffled windows
            drop = rule(a0 - a1)
            if task == 'negative':                # shuffling again must not lose anything
                ok = abs(drop[0]) <= 2 * drop[1] and abs(drop[0]) < NEGATIVE_CONTROL_BOUND
                check = f'drop within 2 sd of 0 and below {NEGATIVE_CONTROL_BOUND} in size'
            else:                                 # with the order gone, only chance is left
                ok = abs(a1.mean() - chance) <= 2 * a1.std(ddof=1)
                check = 'shuffled within 2 sd of chance'
            checks.append((f'{task}: {m} {check}', bool(ok)))
            rows.append({'task': task, 'model': m, 'side': 'order-using', 'in order, %': pct(a0.mean(), a0.std(ddof=1)),
                         'shuffled, %': pct(a1.mean(), a1.std(ddof=1)), 'drop, points (t)': pts(drop), 'check': check,
                         'result': met(ok)})
            if best is None or a0.mean() > best[1]:
                best = (m, a0.mean())

        # The positive control: the best network must find the order
        if task in POSITIVE_CONTROL_BOUND and best is not None:
            ok = bool(best[1] > POSITIVE_CONTROL_BOUND[task])
            checks.append((f'{task}: positive control', ok))
            rows.append({'task': task, 'model': f'best order-using: {best[0]}', 'side': 'positive control',
                         'in order, %': pct(best[1]), 'shuffled, %': '', 'drop, points (t)': '',
                         'check': f'above {POSITIVE_CONTROL_BOUND[task]:.2f}', 'result': met(ok)})
    show_table(rows, 'table5_controls', 'Table 5. The control tasks: answers known in advance.')
    all_met = True
    for _, ok in checks:
        if not ok:
            all_met = False
    controls = {'checks': checks, 'all_met': all_met}

## Table S1. SHD at 25 windows: four order-blind references

Every 25-window network against four references, paired by seed: the ceiling and the best-on-test design
of the designs of 128 to 512 units, and the same two of all the designs (with the one at RadLIF's scale).

In [ ]:
if 'shd_25' in A and 'shd_25_radlif' in A:
    a, b = A['shd_25'], A['shd_25_radlif']        # the 128-unit designs, and all of them
    exp = a['exp']
    na, nb = len(a['candidates']), len(b['candidates'])   # how many candidates each set has
    refs = [(f'ceiling of the {na}', a['ceiling']), (f'best of the {na} on test', a['best_test']),
            (f'ceiling of the {nb}', b['ceiling']), (f'best of the {nb} on test', b['best_test'])]
    rows = []
    for d in list(a['networks'].values()) + list(b['networks'].values()):   # every 25-window network
        row = {'network': d['model'], 'a0, %': pct(d['a0'].mean())}
        for label, design in refs:
            q = rule(d['a0'] - per_seed(exp, design))   # a0 minus this reference, seed by seed
            column = f'against {label} ({design}, {pct(per_seed(exp, design).mean())}%)'
            row[column] = pts(q)
            if q[3]:
                row[column] += ' passes'
        rows.append(row)
    show_table(rows, 'tableS1_candidate_sets', 'Table S1. SHD at 25 windows: a0 minus four order-blind references, in points (t), paired by seed.')

## Table S2. The same networks at two widths on SHD

25 windows against 100 windows of 14 ms, paired by seed (the same seeds and recordings): the change of
accuracy, of the drop and of the gain.

In [ ]:
if 'shd_14ms' in A and 'shd_25' in A:
    fine = A['shd_14ms']                          # 14 ms windows
    rows = []
    pairs = [('ceiling', A['shd_25']), ('rlif', A['shd_25']), ('lstm', A['shd_25']), ('radlif', A.get('shd_25_radlif'))]
    for n, coarse in pairs:                       # coarse: the 25-window setting of this network
        if coarse is None:
            continue
        if n == 'ceiling':                        # the two ceilings
            x, y = coarse['r'], fine['r']
            rows.append({'network': f'ceiling ({coarse["ceiling"]}; {fine["ceiling"]})', '25 windows, %': pct(x.mean()),
                         '14 ms, %': pct(y.mean()), '14 ms minus 25 windows, points (t)': pts(rule(y - x)),
                         'drop difference (t)': '', 'gain difference (t)': '', 'verdicts (25 windows; 14 ms)': ''})
            continue
        if n not in coarse['networks'] or n not in fine['networks']:
            continue                              # this network is missing at one of the widths
        d1, d2 = coarse['networks'][n], fine['networks'][n]
        row = {}
        row['network'] = f'{d1["model"]}; {d2["model"]}'
        row['25 windows, %'] = pct(d1['a0'].mean())
        row['14 ms, %'] = pct(d2['a0'].mean())
        row['14 ms minus 25 windows, points (t)'] = pts(rule(d2['a0'] - d1['a0']))
        row['drop difference (t)'] = pts(rule((d2['a0'] - d2['a1']) - (d1['a0'] - d1['a1'])))
        row['gain difference (t)'] = pts(rule((d2['a0'] - fine['r']) - (d1['a0'] - coarse['r'])))
        row['verdicts (25 windows; 14 ms)'] = f'{d1["verdict"]}; {d2["verdict"]}'
        rows.append(row)
    show_table(rows, 'tableS2_two_widths', 'Table S2. The same networks at two widths on SHD, paired by seed.')

## Table S3. Networks of one setting compared

rlif against rnn (spiking and conventional twins with the same weights), rlif against lif_std (what the
recurrent weights add), and every network against the LSTM: accuracy, drop and energy.

In [ ]:
rows = []
for key in A:
    a = A[key]
    exp, nets = a['exp'], a['networks']
    lstm = a['group']['lstm']                     # the name of the setting's LSTM
    pairs = [('rlif', 'rnn', 'spiking and conventional twins (same weights)'),
             ('rlif', 'lif_std', 'what the recurrent weights add')]
    for n in nets:
        if n != lstm:
            pairs.append((n, lstm, 'against the LSTM'))
    for x, y, what in pairs:
        if x not in nets or y not in nets:
            continue                              # one of the two is not in this setting
        dx, dy = nets[x], nets[y]
        ratio = mean_energy(exp, dx['model']) / mean_energy(exp, dy['model'])
        row = {}
        row['setting'] = a['group']['label']
        row['pair'] = f'{dx["model"]} minus {dy["model"]}'
        row['what it shows'] = what
        row['accuracy difference, points (t)'] = pts(rule(dx['a0'] - dy['a0']))
        row['drop difference, points (t)'] = pts(rule((dx['a0'] - dx['a1']) - (dy['a0'] - dy['a1'])))
        row['energy ratio (worst / within / across)'] = ' / '.join(f'{v:.2f}' for v in ratio)
        rows.append(row)
show_table(rows, 'tableS3_network_pairs', 'Table S3. Networks of one setting compared, paired by seed.')

## Table S4. SHD at 25 windows: shuffling only at test time

A network trained with the windows in order, tested with 25%, 50%, 75% and 100% of every recording's
windows shuffled, without retraining; and, for comparison, a1 (retrained on shuffled windows).

In [ ]:
if 'shd_25' in A:
    a = A['shd_25']
    exp, rows = a['exp'], []
    networks = list(a['networks'].values())       # the 128-unit networks ...
    if 'shd_25_radlif' in A:
        networks += list(A['shd_25_radlif']['networks'].values())   # ... and RadLIF
    for d in networks:
        recs = []                                 # the network's run in every seed
        for s in exp['seeds']:
            recs.append(exp['runs'][(None, d['model'], 0, s)])
        if not all('eval_only' in r for r in recs):
            continue                              # tested only in order
        row = {'network': d['model'], 'in order, %': pct(d['a0'].mean())}
        for share in recs[0]['eval_only']:        # '0.25', '0.5', '0.75', '1.0'
            accs = [r['eval_only'][share] for r in recs]
            row[f'{float(share):.0%} of windows shuffled at test time, %'] = pct(np.mean(accs))
        row['retrained on shuffled windows (a1), %'] = pct(d['a1'].mean())
        rows.append(row)
    show_table(rows, 'tableS4_test_time_shuffling', 'Table S4. SHD at 25 windows: an in-order network tested on shuffled windows without retraining.')

## Table S5. SHD at 25 windows: speakers never heard in training

SHD's test set holds speakers who are not in the training set. The verdicts again, on those speakers alone.

In [ ]:
if 'shd_25' in A and 'test_speakers' in A['shd_25']['exp']['meta']:
    a = A['shd_25']
    exp = a['exp']
    spk = np.array(exp['meta']['test_speakers'])  # the speaker of every test recording
    y = np.array(exp['meta']['test_labels'])      # the class of every test recording
    unheard = ~np.isin(spk, exp['meta']['train_speakers'])    # True for the speakers never heard in training


    def split_acc(model):
        """Per seed: accuracy on the unheard speakers, and on the speakers heard in training."""
        right = []
        for seed in exp['seeds']:
            preds = np.array(exp['runs'][(None, model, 0, seed)]['preds'])
            right.append(preds == y)              # True where the answer is right
        right = np.array(right)                   # one row per seed, one column per test recording
        return right[:, unheard].mean(1), right[:, ~unheard].mean(1)


    r_new, r_old = split_acc(a['ceiling'])        # the ceiling on both groups of speakers
    rows = [{'model': f'ceiling ({a["ceiling"]})', 'unheard speakers, %': pct(r_new.mean()),
             'speakers heard in training, %': pct(r_old.mean()), 'verdict on the unheard speakers alone': '',
             'verdict on the whole test set': ''}]
    for d in a['networks'].values():
        n_new, n_old = split_acc(d['model'])
        row = {}
        row['model'] = d['model']
        row['unheard speakers, %'] = pct(n_new.mean())
        row['speakers heard in training, %'] = pct(n_old.mean())
        row['verdict on the unheard speakers alone'] = verdict(rule(n_new - r_new), d['competent'])
        row['verdict on the whole test set'] = d['verdict']
        rows.append(row)
    show_table(rows, 'tableS5_unheard_speakers',
               f'Table S5. SHD at 25 windows: {int(unheard.sum())} of {len(y)} test recordings come from speakers never heard in training.')

## Table S6. Hidden spikes, and RadLIF's learned time constants

Spikes per test recording, per layer and per neuron per window; for RadLIF, the quartiles of its
learned membrane time constants (in ms) and the epoch kept in every seed.

In [ ]:
rows = []
for key in A:
    a = A[key]
    exp = a['exp']
    T = exp['meta']['windows']                    # windows per recording
    for d in a['networks'].values():
        recs = []                                 # the network's run in every seed
        for s in exp['seeds']:
            recs.append(exp['runs'][(None, d['model'], 0, s)])
        if recs[0].get('spikes_per_recording') is None:
            continue                              # not a spiking network
        spikes = np.mean([r['spikes_per_recording'] for r in recs])   # average over the seeds
        row = {'setting': a['group']['label'], 'network': d['model'], 'hidden spikes per recording': f'{spikes:,.0f}'}
        if recs[0].get('layer_spikes_per_recording'):                 # RadLIF: two hidden layers of 1,024
            layers = np.mean([r['layer_spikes_per_recording'] for r in recs], 0)
            row['per layer'] = ', '.join(f'{v:,.0f}' for v in layers)
            row['per neuron per window'] = ', '.join(f'{v / (1024 * T):.3f}' for v in layers)
        else:                                                          # one layer of 128
            row['per layer'] = ''
            row['per neuron per window'] = f'{spikes / (128 * T):.3f}'
        if 'tau_mem_windows_quartiles' in recs[0]:                     # RadLIF's learned time constants
            q = np.mean([r['tau_mem_windows_quartiles'] for r in recs], 0) * exp['meta']['window_ms']   # windows -> ms
            row['membrane time constants, ms (quartiles per layer)'] = '; '.join(', '.join(f'{v:.0f}' for v in layer) for layer in q)
            row['best validation epochs'] = ', '.join(str(r['best_epoch']) for r in recs)
        else:
            row['membrane time constants, ms (quartiles per layer)'] = ''
            row['best validation epochs'] = ''
        rows.append(row)
show_table(rows, 'tableS6_spikes', 'Table S6. Hidden spikes per test recording; RadLIF\'s learned membrane time constants.')

## Table S7. When is a spiking first layer cheaper?

A conventional first layer reads every (window, channel) cell once (m each) and does a multiply-add
(3m + c) for every non-empty cell and every one of its H neurons. A spiking first layer is charged per
input spike (3m + a per spike and neuron) or per non-empty cell (3m + c). With T windows, C channels, a
share p of non-empty cells and s input spikes per channel, the two cost the same at

```
s* = T (m / H + p (3m + c)) / (3m + a)      (the break-even, in input spikes per channel)
```

Below 1 in the last two columns, the spiking first layer is cheaper. The cell also prints the
break-evens of hidden layers, in spikes per neuron per window.

In [ ]:
rows = []
for key in A:
    a = A[key]
    exp = a['exp']
    cost = exp['meta']['energy_costs_pJ']         # the costs saved with the experiment
    m, c, add = cost['memory_access'], cost['multiply_add'], cost['addition']
    e_spike = 3 * m + add                         # a spike delivered to one synapse
    e_first = 3 * m + c                           # a first-layer multiply-add on a non-empty cell
    ip = exp['meta']['input_per_test_recording']  # the average input of a test recording
    T, Cn = ip['windows'], ip['channels']
    p = ip['nonzero_cells'] / (T * Cn)            # the share of non-empty cells
    s = ip['spikes'] / Cn                         # input spikes per channel
    for d in a['networks'].values():
        if exp['meta']['models'][d['model']]['family'] != 'spiking':
            continue                              # only spiking networks
        if d['model'] == 'radlif':
            h = 1024                              # neurons of the first layer
        else:
            h = 128
        s_star = T * (m / h + p * e_first) / e_spike  # the break-even
        conv = T * Cn * (m + p * h * e_first)     # the conventional first layer's energy
        row = {}
        row['setting'] = a['group']['label']
        row['network'] = d['model']
        row['C'] = Cn
        row['H'] = h
        row['T'] = T
        row['p (non-empty cells)'] = f'{p:.4f}'
        row['s (input spikes per channel)'] = f'{s:.2f}'
        row['s* (break-even)'] = f'{s_star:.2f}'
        row['spiking over conventional, per spike'] = f'{s / s_star:.3f}'           # charged per input spike
        row['per cell'] = f'{p * T * Cn * h * e_first / conv:.3f}'               # charged per non-empty cell
        rows.append(row)
if len(rows) > 0:
    cost = next(iter(A.values()))['exp']['meta']['energy_costs_pJ']   # the costs of the first setting
    m, c, add = cost['memory_access'], cost['multiply_add'], cost['addition']
    print(f'hidden-layer break-evens (spikes per neuron per window): {(4 * m + c) / (3 * m + add):.2f} with no reuse, '
          f'{(m + 4 * c) / (3 * m + add):.2f} within a window, {(4 * c + m / 25) / (2 * m + c + add):.2f} (T = 25) and '
          f'{(4 * c + m / 100) / (2 * m + c + add):.2f} (T = 100) across the windows')
show_table(rows, 'tableS7_first_layer', 'Table S7. A spiking first layer against a conventional one that skips empty cells.')

## Table S8. Every model's energy, by parts

Energy per test recording of every model of every experiment (computed, not measured), and which part
of the computation costs what (first seed).

In [ ]:
rows = []
for name in exps:
    exp = exps[name]
    if name == 'shd_controls':
        continue                                  # the control tasks have no energy
    for mdl in exp['meta']['designs'] + exp['meta']['networks']:
        if not has(exp, mdl):
            continue                              # runs missing
        r = exp['runs'][(None, mdl, 0, exp['seeds'][0])]     # the first seed's run
        e = mean_energy(exp, mdl)
        acc = per_seed(exp, mdl)
        parts = '; '.join(f'{k} {v:,.0f}' for k, v in r['energy_pJ']['parts'].items())   # 'first_layer 265,839; ...'
        row = {}
        row['experiment'] = name
        row['model'] = mdl
        row['family'] = exp['meta']['models'][mdl]['family']
        row['weights'] = exp['meta']['models'][mdl]['weights']
        row['test accuracy, %'] = pct(acc.mean(), acc.std(ddof=1))
        row['worst, uJ'] = uj(e[0])
        row['within a window, uJ'] = uj(e[1])
        row['across the windows, uJ'] = uj(e[2])
        row['worst case is'] = r['energy_pJ']['worst_is']
        row['parts (first seed, pJ)'] = parts
        rows.append(row)
show_table(rows, 'tableS8_energy_by_model', 'Table S8. Every model\'s energy per test recording (computed, not measured).')

## Table S9. The accuracy-energy frontier

In every panel of Figure 3 and every case: the models that no other model beats on both accuracy and
energy, cheapest first. (The best-on-test designs are left out: they are checks, never choices.)
`panel_models` lists the models a panel draws; Figures 3 and S2 use it too.

In [ ]:
def panel_models(panel):
    """The models drawn in one panel: (label, experiment, model, family, role, accuracy per seed)."""
    items, seen = [], []
    for key in panel['groups']:
        if key not in A:
            continue                              # this setting has no results yet
        a = A[key]
        exp = a['exp']
        if has(exp, 'count_256') and 'count_256' not in seen:          # the counts-only floor
            items.append(('counts only', exp, 'count_256', 'counts only', 'floor', per_seed(exp, 'count_256')))
            seen.append('count_256')
        if key in ('shd_25', 'shd_14ms', 'ssc_10ms'):
            label = 'ceiling'
        else:
            label = 'ceiling at RadLIF scale'
        if a['ceiling'] not in seen:                                    # the ceiling
            items.append((label, exp, a['ceiling'], 'order-blind', 'ceiling', a['r']))
            seen.append(a['ceiling'])
        if a['best_test'] not in seen and key != 'shd_25_radlif':       # the best design on test (a check)
            items.append(('best on test', exp, a['best_test'], 'order-blind', 'check', per_seed(exp, a['best_test'])))
            seen.append(a['best_test'])
        for d in a['networks'].values():                                # every network
            if d['model'] not in seen:
                family = exp['meta']['models'][d['model']]['family']
                items.append((star(d), exp, d['model'], family, 'network', d['a0']))
                seen.append(d['model'])
    return items


rows = []
for panel in PANELS:
    items = []
    for item in panel_models(panel):
        if item[4] != 'check':                    # leave out the best-on-test design
            items.append(item)
    if len(items) == 0:
        continue
    for case, idx in (('worst', 0), ('within a window', 1), ('across the windows', 2)):
        points = []                               # (label, energy, accuracy) of every model
        for label, exp, mdl, fam, role, acc in items:
            points.append((label, mean_energy(exp, mdl)[idx], acc.mean()))
        front = []                                # the models no other model beats on both
        for p in points:
            beaten = False
            for q in points:                      # q beats p: as cheap and as accurate, and better in one
                if q[1] <= p[1] and q[2] >= p[2] and (q[1] < p[1] or q[2] > p[2]):
                    beaten = True
            if not beaten:
                front.append(p)
        front.sort(key=lambda p: p[1])            # cheapest first
        text = ' -> '.join(f'{lab} ({100 * acc:.1f}%, {e / 1e6:.3g} uJ)' for lab, e, acc in front)
        rows.append({'panel': panel['title'], 'case': case, 'frontier, cheapest first': text})
show_table(rows, 'tableS9_frontier', 'Table S9. The accuracy-energy frontier of every panel, in each case.')

## Figure 1. The method in one picture

In [ ]:
fig, ax = plt.subplots(figsize=(6.5, 2.5))       # 6.5 x 2.5 inches: the width of a printed page
ax.set_xlim(0, 100)                               # the drawing uses its own units: 100 wide ...
ax.set_ylim(0, 47)                                # ... and 47 high
ax.axis('off')                                    # a diagram: no axes


def box(x, y, w, h, text, edge=INK2, face='white', bold=False):
    """A rounded box with its lower left corner at (x, y), width w, height h, and centred text."""
    ax.add_patch(FancyBboxPatch((x, y), w, h, boxstyle='round,pad=0.3,rounding_size=1.0', fc=face, ec=edge, lw=0.9))
    if bold:
        weight = 'bold'
    else:
        weight = 'normal'
    ax.text(x + w / 2, y + h / 2, text, ha='center', va='center', fontsize=6.8, color=INK, fontweight=weight)


def arrow(x0, y0, x1, y1):
    """An arrow from (x0, y0) to (x1, y1)."""
    ax.annotate('', xy=(x1, y1), xytext=(x0, y0), arrowprops=dict(arrowstyle='-|>', color=INK2, lw=0.8, shrinkA=0, shrinkB=0))


box(0.5, 21, 15, 9, 'a recording:\nspikes on\nmany channels')
box(20.5, 21, 15, 9, 'a table with\none row per\ntime window')
arrow(16.2, 25.5, 19.8, 25.5)
boxes = [(40, 'order-using network,\nwindows in order', 'accuracy a0', BLUE, '#eef4fc'),        # (height, model,
         (27.5, 'the same network, retrained\non shuffled windows', 'accuracy a1', INK2, '#f3f3f1'),  # result, edge colour,
         (15, 'order-blind design: sees every\nwindow, never their order', 'ceiling r', BLUE, '#f1effb')]  # fill colour)
for y, text, result, edge, face in boxes:
    box(43, y - 4, 28, 8, text, edge=edge, face=face)                  # the model
    box(78, y - 3, 16, 6, result, edge=edge, face=face, bold=True)     # its result
    arrow(36.2, 25.5, 42.3, y)                                         # from the table to the model
    arrow(71.7, y, 77.3, y)                                            # from the model to its result
ax.text(68, 3.5, 'drop = a0 − a1 = (a0 − r) + (r − a1) = gain + deficit', ha='center', va='center',
        fontsize=7.2, color=INK, bbox=dict(boxstyle='round,pad=0.45', fc='white', ec=MUTED, lw=0.7))
save_figure(fig, 'figure1_method')

## Figure 2. Every network's loss from scrambling, split at its ceiling

One row per network and setting: a0 (filled dot), a1 (open dot) and the ceiling r (blue line). The
orange bar is the gain (a0 - r), the blue bar the deficit (r - a1). A grey bar marks a network below the
competence bar. The column on the right gives the gain in points (bold when it passes the rule). Each
network is named by its kind, layers and neurons, as in Figure 4: SNN-FF 128 and SNN-R 128 are the
feedforward and recurrent spiking networks of 128 neurons, and RadLIF 2×1,024 has two layers of 1,024.

In [ ]:
rows = []                                         # ('header', setting label, a) or ('net', network, a)
for key in A:
    a = A[key]
    rows.append(('header', a['group']['label'], a))
    for d in a['networks'].values():
        rows.append(('net', d, a))
if len(rows) > 0:
    lows = []                                     # the lowest values drawn, to set the left edge
    for kind, item, a in rows:
        if kind == 'net':
            lows.append(item['a1'].mean())
    for key in A:
        lows.append(A[key]['r'].mean())
    x0 = max(0.0, math.floor((min(lows) - 0.05) * 20) / 20)   # left edge of the axis, a multiple of 0.05
    height = 0.13 * len(rows) + 1.0               # the figure grows with the number of rows (inches)
    fig, ax = plt.subplots(figsize=(5.2, height))   # 5.2 inches wide: fits beside a caption on a page
    text_at = blended_transform_factory(ax.transAxes, ax.transData)   # x in fractions of the axis, y in rows
    y, ticks, labels = len(rows), [], []          # y counts down from the top row
    for kind, item, a in rows:
        y -= 1
        if kind == 'header':                      # a setting's title, in bold
            ax.text(x0 + 0.003, y, item, fontsize=7.5, fontweight='bold', color=INK, va='center')
            continue
        d, r = item, a['r'].mean()
        a0, a1 = d['a0'].mean(), d['a1'].mean()
        ticks.append(y)
        labels.append(nice(d['model']) + ('' if d['competent'] else '*'))   # readable name; * below the bar
        if d['competent']:
            ax.plot([a1, r], [y, y], color=BLUE, lw=3.2, solid_capstyle='butt', zorder=2)                     # deficit
            ax.plot([min(a0, r), max(a0, r)], [y, y], color=ORANGE, lw=3.2, solid_capstyle='butt', zorder=3)  # gain
        else:
            ax.plot([a1, a0], [y, y], color=MUTED, lw=3.2, solid_capstyle='butt', zorder=2)                   # grey bar
        ax.plot([r, r], [y - 0.38, y + 0.38], color=BLUE, lw=1.2, zorder=4)                                 # ceiling
        ax.scatter([a0], [y], s=16, color=INK, zorder=5, edgecolor='white', linewidth=0.4)                  # a0
        ax.scatter([a1], [y], s=16, facecolor='white', edgecolor=INK, linewidth=0.8, zorder=5)              # a1
        g = d['gain']
        if d['competent']:
            txt = minus(f'{100 * g[0]:+.1f}')     # the gain in points
        else:
            txt = 'not interpreted'
        if g[3] and d['competent']:
            weight = 'bold'                       # the gain passes the rule
        else:
            weight = 'normal'
        ax.text(1.01, y, txt, transform=text_at, fontsize=7, va='center', color=INK, fontweight=weight)
    ax.text(1.01, len(rows) - 0.3, 'gain, points', transform=text_at, fontsize=7, color=INK2, va='center')
    ax.set_yticks(ticks)                          # the network names on the left
    ax.set_yticklabels(labels)
    ax.tick_params(axis='y', length=0)
    ax.set_ylim(-0.7, len(rows) - 0.3)
    ax.set_xlim(x0, 1.0)
    ax.grid(axis='y', visible=False)              # vertical grid lines only
    ax.set_xlabel('test accuracy')
    handles = [Line2D([], [], ls='none', marker='o', ms=4, color=INK, label='a0, windows in order'),
               Line2D([], [], ls='none', marker='o', ms=4, mfc='white', mec=INK, label='a1, retrained on shuffled windows'),
               Line2D([], [], color=BLUE, lw=1.2, label='ceiling r'),
               Patch(color=BLUE, label='deficit (r − a1)'), Patch(color=ORANGE, label='gain (a0 − r)')]
    if any(lab.endswith('*') for lab in labels):
        handles.append(Line2D([], [], ls='none', label='* below the competence bar'))
    fig.legend(handles=handles, loc='lower center', ncol=3, frameon=False, bbox_to_anchor=(0.5, 0.0), handlelength=1.4, columnspacing=1.6)
    fig.tight_layout(rect=(0, 0.5 / height, 1, 1))   # leave half an inch at the bottom for the legend
    save_figure(fig, 'figure2_decomposition')

## Figures 3 and S2. Energy against accuracy

One panel per dataset and width; energy per recording on a log scale. Filled markers: the worst case;
hollow markers: the best cases. Every model is named by a label beside its filled marker.

**How the labels are placed.** Labels are placed from the top of a panel to the bottom. Each label tries
the spots in `SPOTS` one after another (just to the right of its marker first) and keeps the
first spot that stays inside the panel, covers no other label and no marker, and is at least 3 points
nearer its own model's markers than any other model's (if no spot does, the spot that comes closest). To pin a label by hand, write for example
`LABEL_SPOT[('SHD, 14 ms windows', 'ceiling')] = (5, -3, 'left', 'top')` after the line that creates
`LABEL_SPOT` and run the cell again.

In [ ]:
# The spots a label may take, in the order they are tried: the (x, y) offset from the marker in points,
# and how the text is aligned there. (6, 0, 'left', 'center') puts the label just to the right, at the
# marker's height.
SPOTS = [(6, 0, 'left', 'center'), (-6, 0, 'right', 'center'), (5, 3, 'left', 'bottom'), (5, -3, 'left', 'top'),
         (-5, 3, 'right', 'bottom'), (-5, -3, 'right', 'top'), (0, 6, 'center', 'bottom'), (0, -6, 'center', 'top'),
         (10, 9, 'left', 'bottom'), (10, -9, 'left', 'top'), (-10, 9, 'right', 'bottom'), (-10, -9, 'right', 'top')]
LABEL_SPOT = {}                                   # labels pinned by hand: (panel title, label) -> one of SPOTS
LABEL_SPOT[('SHD, 25 windows per recording', 'rnn')] = (6, 0, 'left', 'center')   # rnn's and lif_std's markers
                                                  # nearly touch in the results of this study: rnn's label at its own height
LABEL_SPOT[('What it takes to gain accuracy', 'RNN 128')] = (5, -3, 'left', 'top')   # Figure 4: below and right
                                                  # of its own marker, clear of the SNN-R 128 marker next to it


def put(t, spot):
    """Move the label t to a spot."""
    t.xyann = spot[:2]                            # the offset from the marker, in points
    t.set_ha(spot[2])                             # horizontal alignment of the text at that spot
    t.set_va(spot[3])                             # vertical alignment


def distance(box, p):
    """Distance in pixels from the point p to the box of a label (0 when p is inside the box)."""
    dx = max(box.x0 - p[0], 0.0, p[0] - box.x1)
    dy = max(box.y0 - p[1], 0.0, p[1] - box.y1)
    return math.hypot(dx, dy)


def place_labels(fig, labels, points, spots=None):
    """labels: (label, model number) pairs; points: (panel, (x, y), model number) for every marker drawn.
    spots (optional): model number -> the spots that label may use, instead of SPOTS.
    If no spot meets all the conditions, the spot that is inside, free and leans most towards its own
    model is used, and failing that the first spot that may be used."""
    fig.canvas.draw()                             # draw once, so matplotlib knows the size of every label
    r = 3.6 * fig.dpi / 72                        # half the width of a marker (3.6 points), in pixels
    margin = 3 * fig.dpi / 72                     # a label must be 3 points nearer its own model than any other
    marks = []
    for ax, xy, k in points:
        marks.append((ax, ax.transData.transform(xy), k))   # every marker's position in pixels
    placed = []                                   # the boxes of the labels placed so far
    labels = sorted(labels, key=lambda tk: -tk[0].axes.transData.transform(tk[0].xy)[1])   # top to bottom
    for t, k in labels:
        panel = t.axes.get_window_extent()        # the box of the label's panel
        mine, others = [], []                     # the markers of this label's model, and of the others
        for ax, p, j in marks:
            if ax is t.axes and j == k:
                mine.append(p)
            elif ax is t.axes:
                others.append(p)
        chosen = LABEL_SPOT.get((t.axes.get_title(loc='left'), t.get_text()))   # pinned by hand?
        fallback, lean = None, -math.inf          # the inside and free spot that leans most towards its model
        allowed = SPOTS                           # the spots this label may use
        if spots is not None and k in spots:
            allowed = spots[k]
        if chosen is None:
            for spot in allowed:
                put(t, spot)
                box = t.get_window_extent()       # the box the label covers at this spot
                inside = panel.contains(box.x0, box.y0) and panel.contains(box.x1, box.y1)
                free = True
                for b in placed:                  # does it cover a label placed before?
                    if box.overlaps(b):
                        free = False
                for p in mine + others:           # does it cover a marker?
                    if box.overlaps(Bbox.from_bounds(p[0] - r, p[1] - r, 2 * r, 2 * r)):
                        free = False
                if inside and free:
                    nearest_mine = min(distance(box, p) for p in mine)
                    nearest_other = min([distance(box, p) for p in others] + [math.inf])
                    if nearest_other - nearest_mine >= margin:   # the label clearly belongs to its own model
                        chosen = spot
                        break
                    if nearest_other - nearest_mine > lean:      # the best spot so far
                        fallback, lean = spot, nearest_other - nearest_mine
        if chosen is None:
            chosen = fallback
        if chosen is None:
            chosen = allowed[0]
        put(t, chosen)
        placed.append(t.get_window_extent())


def log_axis(ax, xs):
    """A log-scale energy axis wide enough for every value xs."""
    lo, hi = min(xs) / 2, max(xs) * 3             # some room on both sides
    if hi / lo < 10:                              # always show at least one power of ten
        mid = math.sqrt(lo * hi)
        lo, hi = mid / 3.2, mid * 3.2
    ax.set_xscale('log')
    ax.set_xlim(lo, hi)
    ax.xaxis.set_major_locator(LogLocator(base=10))                     # ticks at 0.1, 1, 10, ...
    ax.xaxis.set_major_formatter(FuncFormatter(lambda v, _: f'{v:g}'))   # written as 0.1, 1, 10 (not 10^-1)
    ax.xaxis.set_minor_formatter(NullFormatter())                       # no labels on the small ticks


def energy_panels(name, cases, note):
    """Draw one figure. cases: which of worst (0), within (1), across (2) to draw; the first is drawn
    filled. note: the legend entry that explains the filled and hollow markers."""
    panels = []
    for panel in PANELS:
        items = panel_models(panel)
        if len(items) > 0:                        # skip panels without results
            panels.append((panel, items))
    if len(panels) == 0:
        return
    nrow = math.ceil(len(panels) / 2)             # two panels per row; an odd count leaves room for the legend
    fig, axes = plt.subplots(nrow, 2, figsize=(6.5, 2.6 * nrow), squeeze=False)
    labels, points = [], []                       # every label, and every marker (for place_labels)
    for ax, (panel, items) in zip(axes.flat, panels):
        xs = []                                   # every energy drawn in this panel
        for k, (label, exp, mdl, fam, role, acc) in enumerate(items):   # k numbers the models of a panel
            color, marker = FAMILY_STYLE[fam]
            e = mean_energy(exp, mdl) / 1e6       # pJ -> microjoules
            y = acc.mean()
            for c in cases:
                xs.append(e[c])
            ax.plot([min(e[c] for c in cases), e[cases[0]]], [y, y], color=MUTED, lw=0.7, zorder=1)   # joins the cases
            for j in range(1, len(cases)):        # the hollow markers
                if j == 1 and len(cases) > 2:
                    size = 26                     # Figure S2: 'within' is drawn larger than 'across'
                else:
                    size = 14
                ax.scatter(e[cases[j]], y, s=size, marker=marker, facecolor='white', edgecolor=color, linewidth=0.9, zorder=3)
                points.append((ax, (e[cases[j]], y), k))
            if len(acc) > 1:                      # the spread over the seeds
                ax.errorbar(e[cases[0]], y, yerr=acc.std(ddof=1), fmt='none', ecolor=color, elinewidth=0.8, zorder=2)
            ax.scatter(e[cases[0]], y, s=30, marker=marker, color=color, edgecolor='white', linewidth=0.5, zorder=4)   # filled
            points.append((ax, (e[cases[0]], y), k))
            t = ax.annotate(label, (e[cases[0]], y), xytext=SPOTS[0][:2], textcoords='offset points', fontsize=6.5, color=INK, zorder=5)
            labels.append((t, k))
        log_axis(ax, xs)
        ax.set_title(panel['title'], loc='left', color=INK)
        ax.set_xlabel('energy per recording, µJ (log scale)')
        ax.set_ylabel('test accuracy')
        ax.margins(y=0.12)                        # some room above and below the markers
    spare = list(axes.flat)[len(panels):]         # panels left empty
    for ax in spare:
        ax.axis('off')
    handles = []                                  # the legend: one entry per kind of model, and the note
    for family in FAMILY_STYLE:
        color, marker = FAMILY_STYLE[family]
        handles.append(Line2D([], [], ls='none', marker=marker, ms=5, color=color, label=family))
    handles.append(Line2D([], [], ls='none', marker='o', ms=5, mfc='white', mec=INK2, label=note))
    if len(spare) > 0:
        spare[0].legend(handles=handles, loc='center', frameon=False)   # the legend fills the empty panel
    else:
        fig.legend(handles=handles, loc='lower center', ncol=3, frameon=False, bbox_to_anchor=(0.5, -0.02))
    fig.tight_layout()
    place_labels(fig, labels, points)             # move the labels apart
    save_figure(fig, name)


energy_panels('figure3_energy_frontier', [0, 2], 'filled: worst case; hollow: best case\nacross the windows (computed, not measured)')
energy_panels('figureS2_energy_cases', [0, 1, 2], 'filled: worst case; large hollow: within a\nwindow; small hollow: across the windows')

## Figure 4. The price of accuracy (one picture)

All three settings in one picture: test accuracy against energy per recording (log scale), one colour
per setting. For every setting, the line is its **frontier**: starting from the cheapest model, it
keeps only the models that are more accurate than every cheaper model. No model trained here is above
and to the left of a frontier, so the frontier shows the least energy it took to reach each accuracy.

The dashed line of each colour is that setting's order-blind ceiling (its value is in the legend), the
boundary of what is reachable without the order of the windows. For SHD with 25 windows, the line is the
ceiling of the designs of 128 to 512 units; RadLIF is compared with the ceiling at its own size, the
hollow circle marked "ceiling at RadLIF scale". The part of a frontier above its ceiling is drawn thick: it is the
accuracy that may need the order. Every step that ends above a ceiling is labelled with its price:
"+2.4 points for x3.9 energy" means 2.4 points of accuracy for 3.9 times the energy of the model before.

Filled markers are on a frontier; hollow markers are not (another model is cheaper and at least as
accurate). `CASE_4` picks the energy case: 0 worst case (no data reuse), 1 best within a window, 2 best
across the windows. Energy is computed with the accounting of the training notebooks, not measured. The
best-on-test designs are left out, as in Table S9: they are checks, never choices.

In [ ]:
CASE_4 = 0                                        # 0 worst case, 1 within a window, 2 across the windows
CASE_NAMES = ['worst case, no data reuse', 'best case within a window', 'best case across the windows']
SETTING_COLORS = [BLUE, AQUA, ORANGE]             # one colour per setting (panel of PANELS), the three checked colours
PRICE_SPOTS = [(0, 4, 'center', 'bottom'), (0, -4, 'center', 'top'),     # a price label: just above or
               (0, 11, 'center', 'bottom'), (0, -11, 'center', 'top'),  # below the middle of its step,
               (-8, 4, 'right', 'bottom'), (-8, -4, 'right', 'top'),    # or ending just left of the middle,
               (8, -4, 'left', 'top'), (8, 4, 'left', 'bottom')]        # or starting just right of it


def crossing(a, b, level):
    """The point where the step from a to b crosses the accuracy `level` (energy on a log scale)."""
    f = (level - a[1]) / (b[1] - a[1])            # how far along the step, from 0 to 1
    return (a[0] * (b[0] / a[0]) ** f, level)


fig, ax = plt.subplots(figsize=(7.1, 6.2))        # 7.1 inches wide: the width of a page inside its margins
labels4, points4 = [], []                         # for place_labels: the labels, and every marker
spots4 = {}                                       # the price labels may only use PRICE_SPOTS
k = 0                                             # a number for every label
named = []                                        # (energy, accuracy) of the points already named
legend4 = []
for panel, color in zip(PANELS, SETTING_COLORS):
    models4 = []                                  # (energy in uJ, accuracy in %, model, family, role)
    for label, exp, mdl, fam, role, acc in panel_models(panel):
        if role != 'check':                       # leave out the best-on-test designs
            models4.append((mean_energy(exp, mdl)[CASE_4] / 1e6, 100 * acc.mean(), mdl, fam, role, label))
    if len(models4) == 0:
        continue                                  # this setting has no results yet
    ceiling = 100 * A[panel['groups'][0]]['r'].mean()   # the setting's order-blind ceiling, in %

    # the frontier: cheapest first, keep a model only if it is more accurate than every cheaper one
    front = []
    for m in sorted(models4, key=lambda m: (m[0], -m[1])):   # by energy; equal energy: most accurate first
        if len(front) == 0 or m[1] > front[-1][1]:
            front.append(m)
    ax.plot([m[0] for m in front], [m[1] for m in front], color=color, lw=1.2, zorder=2)

    # the boundary: the order-blind ceiling
    ax.axhline(ceiling, color=color, lw=0.9, ls=(0, (4, 3)), zorder=1)

    # the part of the frontier above the ceiling, and the price of every step that ends there
    for a, b in zip(front[:-1], front[1:]):
        if b[1] <= ceiling:
            continue                              # this step stays below the ceiling
        start = a if a[1] >= ceiling else crossing(a, b, ceiling)
        ax.plot([start[0], b[0]], [start[1], b[1]], color=color, lw=3.4, zorder=3, solid_capstyle='round')
        mid = (math.sqrt(a[0] * b[0]), (a[1] + b[1]) / 2)   # the middle of the step (log scale)
        text = f'+{b[1] - a[1]:.1f} points for ×{b[0] / a[0]:.1f} energy'
        t = ax.annotate(text, mid, xytext=PRICE_SPOTS[0][:2], textcoords='offset points', fontsize=6.2,
                        color=color, fontweight='bold', zorder=3.5,   # above the lines, below the markers
                        path_effects=[patheffects.withStroke(linewidth=2.5, foreground='white')])
        labels4.append((t, k))
        points4.append((ax, mid, k))
        spots4[k] = PRICE_SPOTS
        k += 1

    # the models
    for e, y, mdl, fam, role, label in models4:
        on_front = any(e == m[0] and y == m[1] for m in front)
        ax.scatter([e], [y], s=36 if on_front else 22, marker=FAMILY_STYLE[fam][1],
                   color=color if on_front else 'white', edgecolor=color, linewidth=1.1, zorder=4)
        points4.append((ax, (e, y), k))
        if any(abs(math.log(e / q[0])) < 0.05 and abs(y - q[1]) < 0.3 for q in named):
            k += 1
            continue                              # the same point in another setting: named once
        name = nice(mdl) + (f' ({label})' if role == 'ceiling' else '')   # e.g. "OB 2x128 (ceiling)"
        t = ax.annotate(name, (e, y), xytext=SPOTS[0][:2], textcoords='offset points', fontsize=6.3,
                        color=INK if on_front else INK2, zorder=5,   # above the lines and the markers
                        path_effects=[patheffects.withStroke(linewidth=2.5, foreground='white')])   # readable over lines
        labels4.append((t, k))
        named.append((e, y))
        k += 1
    legend4.append(Line2D([], [], color=color, lw=1.2, marker='o', ms=4,
                          label=f"{panel['title']} (ceiling {ceiling:.1f}%)"))

ax.set_xscale('log')
ax.xaxis.set_major_locator(LogLocator(base=10))
ax.xaxis.set_major_formatter(FuncFormatter(lambda v, _: f'{v:g}'))
ax.xaxis.set_minor_formatter(NullFormatter())
ax.margins(x=0.12, y=0.08)
ax.set_xlabel(f'energy per recording, µJ (log scale; {CASE_NAMES[CASE_4]}; computed, not measured)')
ax.set_ylabel('test accuracy, %')
ax.set_title('What it takes to gain accuracy', loc='left', color=INK)
legend4.append(Line2D([], [], color=INK2, lw=0.9, ls=(0, (4, 3)), label='order-blind ceiling (boundary)'))
legend4.append(Line2D([], [], color=INK2, lw=3.4, label='frontier above the ceiling'))
for fam in FAMILY_STYLE:                          # marker shapes: the kind of model
    legend4.append(Line2D([], [], ls='none', marker=FAMILY_STYLE[fam][1], mfc='white', mec=INK2, label=fam))
legend4.append(Line2D([], [], ls='none', marker='o', mfc='white', mec=INK2, label='hollow: not on the frontier'))
ax.legend(handles=legend4, loc='upper center', bbox_to_anchor=(0.5, -0.11), ncol=4, fontsize=6.4,
          frameon=False)                         # below the plot, in four columns
fig.tight_layout()
place_labels(fig, labels4, points4, spots4)
save_figure(fig, 'figure4_price_of_accuracy')

## Figure S1. Every design and network, from simple to rich

Counts only, every order-blind design from simple to rich, and every network's a0, one panel per
experiment. The chosen ceilings are blue.

In [ ]:
SHORT = {'shd_25': 'SHD, 25 windows', 'shd_14ms': 'SHD, 14 ms windows', 'ssc_10ms': 'SSC, 10 ms windows'}
names = []                                        # the experiments with results
for e in SHORT:
    if e in exps:
        names.append(e)
present = []                                      # every design with every seed in any experiment
for e in names:
    for d in designs_of(exps[e]):
        if d not in present:
            present.append(d)
nets = []                                         # every network with every seed in any experiment
for e in names:
    for m in exps[e]['meta']['networks']:
        if m not in nets and has(exps[e], m):
            nets.append(m)
groups = []                                       # the rows of the figure, group by group
for group in LADDER:
    groups.append([d for d in group if d in present])
extra = []                                        # designs the ladder does not list
for d in present:
    in_ladder = False
    for group in LADDER:
        if d in group:
            in_ladder = True
    if not in_ladder:
        extra.append(d)
groups.append(extra)
groups.append(nets)
Y, order, y = {}, [], 0.0                         # the height of every row (counted down from the top)
for group in groups:
    if len(group) == 0:
        continue
    for n in group:
        Y[n] = -y
        order.append(n)
        y += 1
    y += 0.6                                      # a small gap between groups
if len(order) > 0:
    fig, axes = plt.subplots(1, len(names), figsize=(6.5, 0.12 * len(order) + 1.3), sharey=True, squeeze=False)
    for ax, e in zip(axes[0], names):             # one panel per experiment
        exp = exps[e]
        chosen = {}                               # design -> which ceilings it is
        for key in A:
            if A[key]['exp']['name'] == exp['name']:
                if key.endswith('radlif'):
                    chosen.setdefault(A[key]['ceiling'], []).append('RadLIF ceiling')
                else:
                    chosen.setdefault(A[key]['ceiling'], []).append('ceiling')
        for d in designs_of(exp):                 # the order-blind designs
            acc = per_seed(exp, d)
            is_ceiling = d in chosen
            if is_ceiling:
                color, size = BLUE, 22            # a ceiling: large and blue
            else:
                color, size = MUTED, 8
            if d == 'count_256':
                marker = 's'                      # counts only: a square
            else:
                marker = 'o'
            if len(acc) > 1:
                sd = acc.std(ddof=1)
            else:
                sd = 0
            ax.errorbar(acc.mean(), Y[d], xerr=sd, fmt='none', ecolor=color, elinewidth=0.7)
            ax.scatter(acc.mean(), Y[d], s=size, marker=marker, color=color, zorder=3)
            if is_ceiling:
                ax.axvline(acc.mean(), color=BLUE, lw=0.6, zorder=1)    # a vertical line through the ceiling
                if len(chosen[d]) == 1:
                    text = chosen[d][0]
                else:
                    text = 'both ceilings'
                ax.annotate(text, (acc.mean(), Y[d]), xytext=(5, 0), textcoords='offset points', fontsize=5.5, color=INK, va='center')
        for m in exp['meta']['networks']:         # the order-using networks
            if has(exp, m):
                acc = per_seed(exp, m)
                if len(acc) > 1:
                    sd = acc.std(ddof=1)
                else:
                    sd = 0
                ax.errorbar(acc.mean(), Y[m], xerr=sd, fmt='none', ecolor=ORANGE, elinewidth=0.7)
                ax.scatter(acc.mean(), Y[m], s=14, marker='D', color=ORANGE, zorder=3)
        ax.set_title(SHORT[e], loc='left', fontsize=7.5)
        ax.set_xlabel('test accuracy')
        ax.grid(axis='y', visible=False)
        ax.margins(x=0.15)
    axes[0][0].set_yticks([Y[n] for n in order])  # the model names on the left
    axes[0][0].set_yticklabels(order, fontsize=6)
    handles = [Line2D([], [], ls='none', marker='s', ms=4, color=MUTED, label='counts only'),
               Line2D([], [], ls='none', marker='o', ms=3, color=MUTED, label='order-blind design (mean ± sd)'),
               Line2D([], [], ls='none', marker='o', ms=5, color=BLUE, label='chosen on held-out recordings (a ceiling)'),
               Line2D([], [], ls='none', marker='D', ms=4, color=ORANGE, label='order-using network, a0')]
    fig.legend(handles=handles, loc='lower center', ncol=2, frameon=False, bbox_to_anchor=(0.5, 0.0))
    fig.tight_layout(rect=(0, 0.07, 1, 1))       # leave room at the bottom for the legend
    save_figure(fig, 'figureS1_ladder')

## Summary

Every verdict in one list, and `summary.json` with every number behind them.

In [ ]:
print('Verdicts (H2), and H1 for every competent network whose drop passes the rule:')
h1_tested, h1_fails = [], []
for key in A:
    a = A[key]
    print(f'  {a["group"]["label"]}: ceiling {a["ceiling"]} {pct(a["r"].mean(), a["r"].std(ddof=1))}% '
          f'(chosen among {len(a["candidates"])}); best on test {a["best_test"]}')
    for d in a['networks'].values():
        print(f'     {d["model"]:12s} a0 {pct(d["a0"].mean())}%  gain {pts(d["gain"])}  -> {d["verdict"]}  | H1: {h1_text(d)}')
        if d['h1_eligible']:
            h1_tested.append(d['model'])
            if not d['h1_holds']:
                h1_fails.append(d['model'])
if len(h1_tested) == 0:
    print('H1 overall: not tested (no eligible network)')
elif len(h1_fails) == 0:
    print('H1 overall: supported for every eligible network')
else:
    print('H1 overall: FAILS for', h1_fails)
if controls is not None:
    if controls['all_met']:
        print('Control tasks: every check met')
    else:
        failed = []
        for k, ok in controls['checks']:
            if not ok:
                failed.append(k)
        print('Control tasks: NOT every check met:', ', '.join(failed))

summary = {}                                      # every number behind the verdicts, for summary.json
for key in A:
    a = A[key]
    summary[key] = {'label': a['group']['label'], 'ceiling': a['ceiling'], 'ceiling_per_seed': a['r'].tolist(),
                    'candidates': a['candidates'], 'best_on_test': a['best_test'], 'lstm': a['lstm'], 'networks': {}}
    for n in a['networks']:
        d = a['networks'][n]
        summary[key]['networks'][n] = {'model': d['model'], 'a0': d['a0'].tolist(), 'a1': d['a1'].tolist(),
                                       'train': d['train'], 'competent': d['competent'], 'gain': list(d['gain']),
                                       'drop': list(d['drop']), 'deficit': list(d['deficit']), 'verdict': d['verdict'],
                                       'h1_eligible': d['h1_eligible'], 'h1_holds': d['h1_holds'],
                                       'energy_pJ': mean_energy(a['exp'], d['model']).tolist()}
summary['controls'] = controls
with open(OUT_DIR + 'summary.json', 'w') as f:
    json.dump(summary, f, indent=1, default=str)  # default=str writes anything unusual as text
print('\ntables, figures and summary.json are in', OUT_DIR)

## Where the outputs are

With `RESULTS = 'mine'` they are on your Google Drive. Otherwise they are on this Colab machine, which is
deleted when the session ends, so they are packed into one `.zip` file and offered as a download.

In [ ]:
if RESULTS == 'mine':
    print('The tables and figures are on your Google Drive, in', OUT_DIR)
else:
    archive = shutil.make_archive(OUT_DIR.rstrip('/'), 'zip', OUT_DIR)   # one .zip file of the folder
    print('The tables and figures are in', archive)
    try:
        from google.colab import files        # Colab's tool for saving a file to your computer
        files.download(archive)               # your browser saves the .zip file
    except ImportError:                       # not on Colab: the file stays where it is
        pass